In [ ]:
#
# 1. cleaning data
#
# import the necessaries libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

from datetime import datetime
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import sys, traceback
print("Python:", sys.executable)

try:
    import imblearn
    print("imbalanced-learn version:", imblearn.__version__)
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    print("SMOTE and RandomUnderSampler import OK")
except Exception:
    traceback.print_exc()

import warnings
warnings.filterwarnings("ignore")

fraud_df = pd.read_csv("../data/raw/Fraud_data.csv")
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")
fraud_df.head()

In [ ]:
fraud_df.shape
fraud_df.describe()

In [ ]:
# handle missing values
fraud_df.isnull().sum()

num_cols = fraud_df.select_dtypes(include=np.number).columns
cat_cols = fraud_df.select_dtypes(include="object").columns

fraud_df[num_cols] = fraud_df[num_cols].fillna(fraud_df[num_cols].median())
fraud_df[cat_cols] = fraud_df[cat_cols].fillna(fraud_df[cat_cols].mode().iloc[0])

In [ ]:
# remove duplicate values
fraud_df = fraud_df.drop_duplicates()
num_duplicates = fraud_df.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

In [ ]:
# correct data types
fraud_df["signup_time"] = pd.to_datetime(fraud_df["signup_time"])
fraud_df["purchase_time"] = pd.to_datetime(fraud_df["purchase_time"])
fraud_df["ip_address"] = fraud_df["ip_address"].astype(int)

In [ ]:
#
# 2. EDA 
#

# univariable analysis
fraud_df.describe()

# Histogram with axis labels
ax = fraud_df["purchase_value"].hist(bins=50, color="skyblue", edgecolor="black")

ax.set_xlabel("Purchase Value")      
ax.set_ylabel("Frequency")           
ax.set_title("Distribution of Purchase Value")  


In [ ]:
# bivariante analysis
fraud_df.groupby("class")["purchase_value"].mean()
fraud_df.groupby("class")["age"].mean()

plt.figure(figsize=(6,4))
sns.barplot(x="class", y="purchase_value", data=fraud_df, palette=["green","red"])
plt.xlabel("Transaction Class")
plt.ylabel("Average Purchase Value")
plt.title("Average Purchase Value by Transaction Class")
plt.show()

plt.figure(figsize=(6,4))
sns.barplot(x="class", y="age", data=fraud_df, palette=["green","red"])
plt.xlabel("Transaction Class")
plt.ylabel("Average Age")
plt.title("Average Age by Transaction Class")
plt.show()

plt.figure(figsize=(6,4))
sns.boxplot(x="class", y="purchase_value", data=fraud_df, palette=["green","red"])
plt.xlabel("Transaction Class")
plt.ylabel("Purchase Value")
plt.title("Purchase Value Distribution by Class")
plt.show()

In [ ]:
# class distribution analysis *+(imbalance)
fraud_df["class"].value_counts(normalize=True)

plt.figure(figsize=(6,4))
sns.countplot(x="class", data=fraud_df, palette=["green","red"])
plt.title("Class Distribution: Non-Fraud vs Fraud")
plt.xlabel("Transaction Class")
plt.ylabel("Count")
plt.show()

In [ ]:
#
# 3. Geolocation Integration
#
# convert ipaddress to integer
fraud_df["ip_address"] = fraud_df["ip_address"].astype(int)

In [ ]:
# Range-Based IP to Country Merge
ip_df["lower_bound_ip_address"] = ip_df["lower_bound_ip_address"].astype(int)
ip_df["upper_bound_ip_address"] = ip_df["upper_bound_ip_address"].astype(int)

fraud_df = fraud_df.sort_values("ip_address")
ip_df = ip_df.sort_values("lower_bound_ip_address")

fraud_df["country"] = pd.merge_asof(
    fraud_df,
    ip_df,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)["country"]


In [ ]:
# Fraud Analysis by Country
fraud_df.groupby("country")["class"].mean().sort_values(ascending=False)

country_fraud = fraud_df.groupby("country")["class"].mean().sort_values(ascending=False)

plt.figure(figsize=(10,5))
sns.barplot(x=country_fraud.index, y=country_fraud.values, palette="Reds_r")
plt.xticks(rotation=45)
plt.ylabel("Fraud Rate")
plt.xlabel("Country")
plt.title("Average Fraud Rate by Country")
plt.show()

In [ ]:
#
# 4. Feature Engineering
#
# time based feature
fraud_df["hour_of_day"] = fraud_df["purchase_time"].dt.hour
fraud_df["day_of_week"] = fraud_df["purchase_time"].dt.dayofweek

fraud_df["time_since_signup"] = (
    fraud_df["purchase_time"] - fraud_df["signup_time"]
).dt.total_seconds() / 3600

In [ ]:
fraud_df["purchase_time"] = pd.to_datetime(fraud_df["purchase_time"], errors="coerce")
fraud_df["hour_of_day"] = fraud_df["purchase_time"].dt.hour
fraud_df["day_of_week"] = fraud_df["purchase_time"].dt.dayofweek

# Calculate average fraud rate by hour of day
hourly_fraud = fraud_df.groupby("hour_of_day")["class"].mean()

In [ ]:
# Transaction Frequency & Velocity
fraud_df = fraud_df.sort_values(["user_id", "purchase_time"])

# Plot
hourly_fraud = fraud_df.groupby("hour_of_day")["class"].mean()

plt.figure(figsize=(8,4))
sns.lineplot(x=hourly_fraud.index, y=hourly_fraud.values, marker="o")
plt.xlabel("Hour of Day")
plt.ylabel("Average Fraud Rate")
plt.title("Fraud Rate by Hour of Day")
plt.show()



In [ ]:
# Transaction Frequency & Velocity
fraud_df = fraud_df.sort_values(["user_id", "purchase_time"])

fraud_df["txn_count_24h"] = (
    fraud_df
    .groupby("user_id")["purchase_time"]
    .transform(lambda x: x.diff().dt.total_seconds().lt(86400).cumsum())
)

In [ ]:
#
# 5. Data Transformation
#
# Feature Scaling
from sklearn.preprocessing import StandardScaler

scale_cols = ["purchase_value", "time_since_signup", "txn_count_24h"]
scaler = StandardScaler()
fraud_df[scale_cols] = scaler.fit_transform(fraud_df[scale_cols])

In [ ]:
print(fraud_df.columns.tolist())

In [ ]:
#  Handle Class Imbalance
#Train-Test Split

from sklearn.model_selection import train_test_split
X = fraud_df.drop("class", axis=1)
y = fraud_df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
# Apply SMOTE (Training Data Only)
from imblearn.over_sampling import SMOTE

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before SMOTE
sns.countplot(x=y_train, ax=axes[0], palette=["green", "red"])
axes[0].set_title("Before SMOTE")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")

# Apply SMOTE
smote = SMOTE(random_state=42)

# After SMOTE
axes[1].set_title("After SMOTE")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### Safer approach: undersample + SMOTE inside an imblearn Pipeline
This cell drops identifier/time columns, scales numeric features only (via ColumnTransformer), applies RandomUnderSampler followed by SMOTE during training, and evaluates on the untouched test set. Use SMOTENC if you have non-one-hot categorical features.